# 01 — CU2 baseline: TF-IDF + Regresión logística

Modelo simple de referencia para texto. La meta del T1 (OE2) es que BETO
(notebook 02) **supere** a este baseline, no solo que llegue a F1-macro ≥
0,85 — por eso el baseline sigue siendo una regresión logística, pero
bien hecha: TF-IDF de palabras (1-2 gramas) y de caracteres (2-5 gramas),
`C` elegido en validación y pesos de clase balanceados.

Corre `entrenar_cu2_tfidf.py` (la misma lógica que en una PC local, ver
`docs/ENTRENAMIENTO_LOCAL.md`). Tarda unos minutos y no necesita GPU.

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {CARPETA_ML}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): la carpeta ml/ del repo.
    CARPETA_ML = str(Path.cwd())

os.chdir(CARPETA_ML)
sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
MODO_RAPIDO = False  # True = pocas filas y 1 época: solo para probar que corre (métricas sin valor)

if MODO_RAPIDO:
    # Si todavía no corriste 00_preparar_dataset, genera un dataset sintético
    # diminuto para poder probar la plomería (no sirve para medir nada).
    from _modo_rapido_fallback import generar_si_falta
    generar_si_falta(Path(CARPETA_ML) / "datos")

import torch
print("GPU disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Sin GPU: en Colab, activa Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")


## Entrenar y evaluar en el conjunto de PRUEBA

In [ ]:
get_ipython().system("python entrenar_cu2_tfidf.py")


## Resultados

In [ ]:
import json
from IPython.display import Image as MostrarImagen, display

carpeta = Path(CARPETA_ML) / "modelos" / "cu2-tfidf-logreg-v1"
metricas = json.loads((carpeta / "metricas.json").read_text(encoding="utf-8"))
print({k: v for k, v in metricas.items() if k in ("accuracy", "f1_macro", "f1_weighted", "n")})
print("Por fuente:", json.dumps(metricas.get("por_fuente", {}), indent=2, ensure_ascii=False))
for nombre in ("curvas_entrenamiento.png", "matriz_confusion.png"):
    if (carpeta / nombre).exists():
        display(MostrarImagen(filename=str(carpeta / nombre)))
